benchmark_2.py — Notebook đơn giản: train + đánh giá model Hybrid (AI-Rec)

Mục đích
────────
Bản RÚT GỌN của `evaluation/benchmark.py`, viết theo dạng "notebook" (từng ô
`# %%`) để chạy tương tác trong VS Code / Cursor (Jupyter extension) hoặc chạy
thẳng bằng `python`. KHÔNG quét lưới siêu tham số — chỉ dùng MỘT cấu hình (lấy
từ `config.py`, đúng cấu hình đang phục vụ) để minh hoạ toàn bộ luồng:

    đọc dữ liệu /data -> chia train/valid/test theo thời gian -> train model
    Hybrid (models/hybrid.py) trên train -> đánh giá trên test, in chỉ số.

Muốn tinh chỉnh siêu tham số (half-life, k, γ, ...) và có đầy đủ baseline/đối
chứng/khoảng tin cậy thì dùng `evaluation/benchmark.py`.

Chạy
────
  bash evaluation/export_data.sh          # xuất dữ liệu vào evaluation/data/
  python -m evaluation.benchmark_2        # từ thư mục ai-rec/ (host hoặc docker)

Cách chia dữ liệu
─────────────────
Cắt theo THỜI GIAN TOÀN CỤC (không theo từng khách — xem benchmark.py docstring
để biết lý do): xếp mọi đơn theo thời gian, 70% đơn đầu = train, 15% kế = valid,
15% cuối = test. Tập `valid` được GIỮ RIÊNG (không gộp vào train) đúng theo yêu
cầu 3 tập — ở bản rút gọn này nó không dùng để tinh chỉnh gì (vì không quét
lưới), chỉ để tách rời hẳn khỏi dữ liệu train/test.

## 1. Import + cấu hình đường dẫn

In [33]:
from __future__ import annotations

import logging
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# `__file__` không tồn tại khi chạy trong notebook (Jupyter) — dùng cwd của
# kernel làm mốc thay thế, để vẫn `from models...` / `from config import ...`
# được dù mở notebook từ ai-rec/evaluation/ hay từ nơi khác.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd().resolve()
    if SCRIPT_DIR.name != "evaluation" and (SCRIPT_DIR / "evaluation").exists():
        SCRIPT_DIR = SCRIPT_DIR / "evaluation"
PROJECT_ROOT = SCRIPT_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (  # noqa: E402
    CB_ONBOARDING_PSEUDO_COUNT, CF_INPUT_SCALE, HYBRID_ALLOW_REPEAT, HYBRID_CF_SHRINK_K,
    HYBRID_REPEAT_WEIGHT, RECENCY_HALF_LIFE_DAYS, REPEAT_SCORE,
)
from models.cf_recommendation import CollaborativeFiltering  # noqa: E402
from models.content_based import build_candidates  # noqa: E402
from models.hybrid import HybridParams, build_recommender  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("benchmark_2")

DATA_DIR = SCRIPT_DIR / "data"

## 2. Tham số chạy

Dùng ĐÚNG cấu hình đang phục vụ (`config.py`) — không quét lưới ở bản này.

In [34]:
MIN_BUYERS = 3              # phải khớp min_buyers_per_product của dbt
CANDIDATES_PER_CELL = 20    # phải khớp cb_candidates_per_cell của dbt
K_VALUES = (5, 10, 20)
MAIN_K = 10

VAL_FRAC = 0.15
TEST_FRAC = 0.15

HALF_LIFE_DAYS = RECENCY_HALF_LIFE_DAYS
CF_SCALE = CF_INPUT_SCALE
REPEAT_CHANNEL = REPEAT_SCORE  # "order_freq" | "cf_weight"

HYBRID_PARAMS = HybridParams(
    cf_shrink_k=HYBRID_CF_SHRINK_K,
    repeat_weight=HYBRID_REPEAT_WEIGHT,
    allow_repeat=HYBRID_ALLOW_REPEAT,
)

logger.info(
    "Cấu hình: half-life=%.0f ngày, thang=%s, kênh mua lại=%s, k=%.2g, γ=%.2g, allow_repeat=%s",
    HALF_LIFE_DAYS, CF_SCALE, REPEAT_CHANNEL,
    HYBRID_PARAMS.cf_shrink_k, HYBRID_PARAMS.repeat_weight, HYBRID_PARAMS.allow_repeat,
)

17:06:45 [INFO] Cấu hình: half-life=37 ngày, thang=log, kênh mua lại=order_freq, k=0, γ=1, allow_repeat=True


## 3. Đọc dữ liệu từ `evaluation/data/`

- `order_lines.csv`: lịch sử mua (bắt buộc)
- `catalog_features.csv`: đặc trưng sản phẩm cho content-based (bắt buộc)
- `preference_signals.csv`: tuỳ chọn onboarding (tuỳ chọn, rỗng nếu không có)

In [35]:
def load_inputs(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    lines = pd.read_csv(
        data_dir / "order_lines.csv",
        dtype={"CustomerID": str, "ProductID": str, "OrderID": str, "Quantity": "int64"},
        parse_dates=["OrderDate"],
    )
    catalog = pd.read_csv(
        data_dir / "catalog_features.csv",
        dtype={"ProductID": str, "ParentCategory": str, "Brand": str, "PriceBand": str},
    )
    sig_path = data_dir / "preference_signals.csv"
    signals = (
        pd.read_csv(sig_path, dtype={"CustomerID": str, "Dimension": str, "Value": str})
        if sig_path.exists() else pd.DataFrame(columns=["CustomerID", "Dimension", "Value", "Polarity"])
    )
    return lines, catalog, signals


lines, catalog, signals = load_inputs(DATA_DIR)
logger.info(
    "Đã đọc dữ liệu: %d dòng đơn hàng, %d sản phẩm, %d tín hiệu onboarding.",
    len(lines), len(catalog), len(signals),
)

17:06:54 [INFO] Đã đọc dữ liệu: 6373096 dòng đơn hàng, 600000 sản phẩm, 77339 tín hiệu onboarding.


## 4. Chia train / valid / test theo thời gian

Cắt theo SỐ ĐƠN (không theo số ngày — dữ liệu có thể dồn cục ở cuối):
xếp mọi đơn theo thời gian đặt, 70% đầu = train, 15% kế = valid, 15% cuối = test.
Ba tập không chồng lấp thời gian với nhau.

In [36]:
def cut_at(df: pd.DataFrame, frac: float) -> pd.Timestamp:
    orders = df.groupby("OrderID")["OrderDate"].min().sort_values(kind="stable")
    return orders.iloc[int(round(len(orders) * frac))]


def ground_truth(df: pd.DataFrame) -> dict[str, set[str]]:
    return df.groupby("CustomerID")["ProductID"].agg(lambda s: set(s)).to_dict()


t_valid = cut_at(lines, 1 - VAL_FRAC - TEST_FRAC)
t_test = cut_at(lines, 1 - TEST_FRAC)

train_lines = lines[lines["OrderDate"] < t_valid]
valid_lines = lines[(lines["OrderDate"] >= t_valid) & (lines["OrderDate"] < t_test)]
test_lines = lines[lines["OrderDate"] >= t_test]

valid_truth = ground_truth(valid_lines)
test_truth = ground_truth(test_lines)

logger.info(
    "Mốc cắt: valid từ %s, test từ %s | đơn train=%d valid=%d test=%d | khách valid=%d test=%d",
    t_valid, t_test,
    train_lines["OrderID"].nunique(), valid_lines["OrderID"].nunique(), test_lines["OrderID"].nunique(),
    len(valid_truth), len(test_truth),
)

17:07:05 [INFO] Mốc cắt: valid từ 2026-07-02 20:25:51, test từ 2026-08-22 12:34:42 | đơn train=1968957 valid=421920 test=421919 | khách valid=36809 test=38024


### 4.1 Chẩn đoán: vì sao MAP/NDCG có thể GIẢM khi K tăng

Công thức `average_precision`/`ndcg` ở mục 7 dùng mẫu số **chặn ở `min(|R_u|, K)`**
(chuẩn Järvelin & Kekäläinen 2002 / trec_eval). Hệ quả:

- Nếu `|R_u| ≤ K` (khách mua ít hơn hoặc bằng K món): mẫu số không đổi khi K
  tăng thêm → MAP/NDCG chỉ có thể TĂNG hoặc giữ nguyên.
- Nếu `|R_u| > K` (khách mua NHIỀU HƠN K món, ví dụ |R|=18 nhưng K=10): mẫu số
  = K, tăng theo K dù model không tìm thêm được hit tương xứng → MAP/NDCG có
  thể GIẢM khi K tăng.

Ô dưới in phân phối thật của `|R_u|` (số món khác nhau khách mua ở test) để
biết đang rơi vào trường hợp nào — **không phải lỗi công thức**, chỉ là bản
chất của biến thể "capped" AP@K/NDCG@K khi ground truth lớn hơn K.

In [37]:
sizes = pd.Series({u: len(v) for u, v in test_truth.items()}, name="n_truth")
print("Phân phối |R_u| (số món khác nhau khách mua ở TEST):")
print(sizes.describe())
print()
for k in K_VALUES:
    pct = (sizes > k).mean()
    print(f"% khách có |R_u| > K={k:<2d}: {pct:6.1%}  (mẫu số AP/IDCG bị chặn ở K thay vì |R_u|)")

Phân phối |R_u| (số món khác nhau khách mua ở TEST):
count    38024.000000
mean        21.854566
std         17.095432
min          1.000000
25%          9.000000
50%         18.000000
75%         30.000000
max        184.000000
Name: n_truth, dtype: float64

% khách có |R_u| > K=5 :  85.9%  (mẫu số AP/IDCG bị chặn ở K thay vì |R_u|)
% khách có |R_u| > K=10:  70.4%  (mẫu số AP/IDCG bị chặn ở K thay vì |R_u|)
% khách có |R_u| > K=20:  43.1%  (mẫu số AP/IDCG bị chặn ở K thay vì |R_u|)


## 5. Dựng tập đầu vào cho model (tái hiện đúng cách serving xử lý)

- `training_frame`: cộng dồn (khách, món) trong TRAIN, lọc món có < `MIN_BUYERS`
  người mua, tính `DaysSince` tới mốc cắt (dùng để train).
- `build_candidates`: tập ứng viên content-based — chỉ dùng doanh số TRONG
  cửa sổ train, không rò rỉ doanh số của valid/test.

In [38]:
def training_frame(df: pd.DataFrame, cut: pd.Timestamp, min_buyers: int) -> pd.DataFrame:
    pairs = (
        df.groupby(["CustomerID", "ProductID"], as_index=False)
        .agg(Quantity=("Quantity", "sum"), LastPurchasedAt=("OrderDate", "max"))
    )
    pairs = pairs[pairs["Quantity"] > 0]
    buyers = pairs.groupby("ProductID")["CustomerID"].nunique()
    pairs = pairs[pairs["ProductID"].isin(buyers[buyers >= min_buyers].index)]
    pairs["DaysSince"] = (cut.normalize() - pairs["LastPurchasedAt"].dt.normalize()).dt.days
    return pairs[["CustomerID", "ProductID", "Quantity", "DaysSince"]].reset_index(drop=True)


def purchase_events_frame(df: pd.DataFrame, cut: pd.Timestamp) -> pd.DataFrame:
    """Một dòng cho mỗi (khách, món, đơn) — dùng cho kênh mua lại `order_freq`."""
    ev = (
        df.groupby(["CustomerID", "ProductID", "OrderID"], as_index=False)
        .agg(Quantity=("Quantity", "sum"), OrderedAt=("OrderDate", "min"))
    )
    ev = ev[ev["Quantity"] > 0]
    ev["DaysSince"] = (cut.normalize() - ev["OrderedAt"].dt.normalize()).dt.days
    return ev[["CustomerID", "ProductID", "DaysSince"]].reset_index(drop=True)


train_frame = training_frame(train_lines, t_valid, MIN_BUYERS)
train_units = train_lines.groupby("ProductID")["Quantity"].sum()
train_candidates = build_candidates(catalog, train_units, CANDIDATES_PER_CELL)
train_events = purchase_events_frame(train_lines, t_valid) if REPEAT_CHANNEL == "order_freq" else None

logger.info(
    "Tập train cho model: %d cặp (khách, món), %d ứng viên content-based.",
    len(train_frame), len(train_candidates),
)

17:07:13 [INFO] Tập train cho model: 2936122 cặp (khách, món), 15107 ứng viên content-based.


## 6. Train model Hybrid

`build_recommender` (models/hybrid.py) gói lại: CF item-based (fit trên
`train_frame`) + content-based (fit trên `train_candidates`) + kênh mua lại
(fit trên `train_events` nếu có) + tham số trộn `HYBRID_PARAMS`.

In [39]:
started = time.monotonic()

cf = CollaborativeFiltering(half_life_days=HALF_LIFE_DAYS, input_scale=CF_SCALE).fit(train_frame)
hybrid = build_recommender(
    cf=cf,
    features=train_candidates,
    signals=signals,
    params=HYBRID_PARAMS,
    onboarding_pseudo_count=CB_ONBOARDING_PSEUDO_COUNT,
    purchase_events=train_events,
)

logger.info("Train xong trong %.1fs. %s", time.monotonic() - started, hybrid.summary())

17:07:21 [INFO] Train xong trong 7.5s. {'customers': 36795, 'products': 14914, 'interactions': 2936122, 'profile_customers': 36795, 'profile_products': 14914, 'behavior_pairs': 0, 'candidates': 15107, 'onboarded_customers': 12066, 'content_based': 1, 'repeat_model': 1, 'behavior': 0}


In [40]:
# ═════════════════════════════════════════════════════════════════════════
# 6.1 Đo hiệu năng hệ thống — 8 chỉ số: Inference/P50/P95 Latency, Throughput,
# Training Time, Memory Usage, CPU Utilization, Error Rate. Ô này TỰ CHỨA
# (không phụ thuộc biến phụ trợ ở ô khác), và train một bản hybrid RIÊNG
# (`_bench_hybrid`) để không đụng biến `hybrid` dùng ở mục 8/9.
#
#   Inference Latency : thời gian TRUNG BÌNH để model tạo 1 danh sách gợi ý (ms)
#   P50 / P95 Latency  : phân vị 50%/95% của latency trên — 95% yêu cầu không
#                        chậm hơn P95 (ms)
#   Throughput         : số yêu cầu gợi ý xử lý được trong 1 giây, đo tuần tự
#                        1 luồng (giống 1 worker API) (requests/s)
#   Training Time      : thời gian train lại CF + Hybrid một lần độc lập (giây)
#   Memory Usage       : ĐỈNH RAM (RSS) của tiến trình trong SUỐT quá trình
#                        train + inference dưới đây — lấy mẫu mỗi 50ms (MB)
#   CPU Utilization    : % CPU tiến trình dùng = cpu_time / wall_time / số lõi
#                        logical * 100, trên CẢ khoảng train + inference (%)
#   Error Rate         : % lần gọi recommend() NÉM LỖI — mẫu gồm cả ID khách
#                        GIẢ (không tồn tại) để có trường hợp lỗi thực tế (%)
import random
import threading

import psutil

N_SAMPLE = 500    # số khách THẬT dùng để đo latency/throughput/error rate
N_UNKNOWN = 20    # số ID khách GIẢ (không tồn tại) chèn vào để đo error rate
RNG_SEED = 42

rng = random.Random(RNG_SEED)
known_sample = rng.sample(sorted(test_truth), min(N_SAMPLE, len(test_truth)))
unknown_sample = [f"__UNKNOWN_CUSTOMER_{i}__" for i in range(N_UNKNOWN)]
bench_requests = known_sample + unknown_sample
rng.shuffle(bench_requests)

LOGICAL_CORES = psutil.cpu_count(logical=True) or 1
_proc = psutil.Process()

# Lấy mẫu RAM song song bằng 1 thread phụ, chạy suốt khối đo dưới đây để bắt
# ĐỈNH RSS thay vì chỉ RAM lúc kết thúc.
_peak_rss = _proc.memory_info().rss
_stop_sampling = threading.Event()


def _sample_rss() -> None:
    global _peak_rss
    while not _stop_sampling.is_set():
        try:
            _peak_rss = max(_peak_rss, _proc.memory_info().rss)
        except Exception:  # noqa: BLE001 - tiến trình có thể đổi trạng thái giữa lúc lấy mẫu
            pass
        _stop_sampling.wait(0.05)


_sampler = threading.Thread(target=_sample_rss, daemon=True)
_sampler.start()

_cpu_start = _proc.cpu_times()
_wall_start = time.perf_counter()

# 1) Training Time — train lại một bản hybrid ĐỘC LẬP để đo thời gian xây dựng.
_train_start = time.perf_counter()
_bench_cf = CollaborativeFiltering(half_life_days=HALF_LIFE_DAYS, input_scale=CF_SCALE).fit(train_frame)
_bench_hybrid = build_recommender(
    cf=_bench_cf, features=train_candidates, signals=signals, params=HYBRID_PARAMS,
    onboarding_pseudo_count=CB_ONBOARDING_PSEUDO_COUNT, purchase_events=train_events,
)
training_time_s = time.perf_counter() - _train_start

# 2) Inference Latency / P50 / P95 / Throughput / Error Rate — gọi TUẦN TỰ.
latencies_ms: list[float] = []
n_errors = 0
for cid in bench_requests:
    t0 = time.perf_counter()
    try:
        _bench_hybrid.recommend(cid, MAIN_K)
    except Exception:  # noqa: BLE001 - mục đích là ĐẾM lỗi, không phân loại
        n_errors += 1
    else:
        latencies_ms.append((time.perf_counter() - t0) * 1000.0)
_inference_wall_s = sum(latencies_ms) / 1000.0

_wall_total_s = time.perf_counter() - _wall_start
_cpu_end = _proc.cpu_times()
_stop_sampling.set()
_sampler.join(timeout=1.0)

cpu_time_s = (_cpu_end.user - _cpu_start.user) + (_cpu_end.system - _cpu_start.system)
cpu_utilization_pct = (
    min(100.0, cpu_time_s / _wall_total_s / LOGICAL_CORES * 100.0) if _wall_total_s > 0 else 0.0
)
memory_usage_mb = _peak_rss / (1024 ** 2)

lat_arr = np.array(latencies_ms) if latencies_ms else np.array([0.0])
inference_latency_ms = float(lat_arr.mean())
p50_latency_ms = float(np.percentile(lat_arr, 50))
p95_latency_ms = float(np.percentile(lat_arr, 95))
throughput_rps = len(latencies_ms) / _inference_wall_s if _inference_wall_s > 0 else float("inf")
error_rate_pct = n_errors / len(bench_requests) * 100.0

perf_table = pd.DataFrame([
    {"Chỉ số": "Inference Latency", "Giá trị": round(inference_latency_ms, 2), "Đơn vị": "ms"},
    {"Chỉ số": "P50 Latency",       "Giá trị": round(p50_latency_ms, 2),       "Đơn vị": "ms"},
    {"Chỉ số": "P95 Latency",       "Giá trị": round(p95_latency_ms, 2),       "Đơn vị": "ms"},
    {"Chỉ số": "Throughput",        "Giá trị": round(throughput_rps, 2),       "Đơn vị": "requests/s"},
    {"Chỉ số": "Training Time",     "Giá trị": round(training_time_s, 2),     "Đơn vị": "giây"},
    {"Chỉ số": "Memory Usage",      "Giá trị": round(memory_usage_mb, 1),     "Đơn vị": "MB"},
    {"Chỉ số": "CPU Utilization",   "Giá trị": round(cpu_utilization_pct, 1), "Đơn vị": "%"},
    {"Chỉ số": "Error Rate",        "Giá trị": round(error_rate_pct, 2),      "Đơn vị": "%"},
]).set_index("Chỉ số")

print(f"\n=== Hiệu năng hệ thống (mẫu {len(known_sample)} khách thật + "
      f"{len(unknown_sample)} ID giả, lõi CPU logical: {LOGICAL_CORES}) ===")
print(perf_table.to_string())


=== Hiệu năng hệ thống (mẫu 500 khách thật + 20 ID giả, lõi CPU logical: 10) ===
                   Giá trị      Đơn vị
Chỉ số                                
Inference Latency     6.52          ms
P50 Latency           7.09          ms
P95 Latency           7.48          ms
Throughput          153.31  requests/s
Training Time         7.33        giây
Memory Usage       3285.60          MB
CPU Utilization      10.10           %
Error Rate            0.00           %


## 7. Hàm chỉ số đánh giá

Precision/Recall/HitRate/MAP/NDCG@K, relevance nhị phân — định nghĩa giống
`evaluation/benchmark.py` (xem docstring ở đó cho nguồn tham khảo).

In [41]:
METRICS = ("Precision", "Recall", "HitRate", "MAP", "NDCG")


def average_precision(relevant: set[str], ranked: list[str], k: int) -> float:
    if not relevant:
        return 0.0
    hits, total = 0, 0.0
    for i, item in enumerate(ranked[:k], start=1):
        if item in relevant:
            hits += 1
            total += hits / i
    return total / min(len(relevant), k)


def ndcg(relevant: set[str], ranked: list[str], k: int) -> float:
    dcg = sum(1.0 / math.log2(i + 2) for i, item in enumerate(ranked[:k]) if item in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg else 0.0


def user_metrics(relevant: set[str], ranked: list[str], k: int) -> dict[str, float]:
    hits = len(relevant & set(ranked[:k]))
    return {
        "Precision": hits / k,
        "Recall": hits / len(relevant),
        "HitRate": float(hits > 0),
        "MAP": average_precision(relevant, ranked, k),
        "NDCG": ndcg(relevant, ranked, k),
    }


def evaluate(hybrid, truth: dict[str, set[str]], k_values: tuple[int, ...]) -> pd.DataFrame:
    """Trả về DataFrame: hàng = K, cột = chỉ số trung bình trên khách có ground truth."""
    top_n = max(k_values)
    users = sorted(truth)
    ranked = {u: [p for p, _, _ in hybrid.recommend(u, top_n)] for u in users}

    rows = []
    for k in k_values:
        per_user = [user_metrics(truth[u], ranked[u], k) for u in users]
        df = pd.DataFrame(per_user, columns=METRICS)
        rows.append({"K": k, "n_users": len(users), **{m: float(df[m].mean()) for m in METRICS}})
    return pd.DataFrame(rows).set_index("K")

## 8. Đánh giá trên tập TEST

Model train trên `train_lines` (trước mốc valid), đánh giá ground truth là
món khách THỰC SỰ mua ở `test_lines`. Ghi chỉ số ra `evaluation/results/` và
in ra màn hình.

In [42]:
started_eval = time.monotonic()
test_metrics = evaluate(hybrid, test_truth, K_VALUES)
logger.info("Đánh giá xong trong %.1fs.", time.monotonic() - started_eval)

print("\n=== Chỉ số trên tập TEST ===")
print(f"Số khách có ground truth: {len(test_truth)}")
print(test_metrics.round(4).to_string())

17:11:51 [INFO] Đánh giá xong trong 259.7s.



=== Chỉ số trên tập TEST ===
Số khách có ground truth: 38024
    n_users  Precision  Recall  HitRate     MAP    NDCG
K                                                      
5     38024     0.1562  0.0423   0.5597  0.1039  0.1775
10    38024     0.1104  0.0602   0.6501  0.0649  0.1429
20    38024     0.0789  0.0854   0.7357  0.0452  0.1237


## 9. (Tham khảo) Đánh giá trên tập VALID

Tập valid được giữ riêng, không gộp vào train — ở đây dùng để xem model có
ổn định giữa hai giai đoạn hay không (KHÔNG dùng số này để chọn tham số, vì
bản rút gọn không quét lưới).

In [43]:
# valid_metrics = evaluate(hybrid, valid_truth, K_VALUES)
# print("\n=== Chỉ số trên tập VALID (tham khảo, không dùng để chọn tham số) ===")
# print(f"Số khách có ground truth: {len(valid_truth)}")
# print(valid_metrics.round(4).to_string())

## 10. Ghi kết quả ra file

In [44]:
# from datetime import datetime, timezone

# RESULTS_DIR = SCRIPT_DIR / "results"
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
# out_path = RESULTS_DIR / f"benchmark_2_{tdd3s}.csv"
# test_metrics.assign(split="test").reset_index().to_csv(out_path, index=False)
# logger.info("Đã ghi %s", out_path)